In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()

if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("Please set OPENAI_API_KEY environment variable")
else:
    llm = ChatOpenAI(model="gpt-5-nano")
    # output = llm.invoke("I want to know the meaning of water").content
    print("OpenAI Key is set")

OpenAI Key is set


In [3]:
llm

ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x10d2d26c0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10d2d38c0>, root_client=<openai.OpenAI object at 0x10d185910>, root_async_client=<openai.AsyncOpenAI object at 0x10d2d2ff0>, model_name='gpt-5-nano', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [5]:
# use duckduck-go
from langchain_community.tools import DuckDuckGoSearchRun
# duck = DuckDuckGoSearchRun()
# print(duck.invoke("What is the meaning of life?"))
from ddgs import DDGS

with DDGS() as ddgs:
    results = list(ddgs.text("meaning of life", max_results=1))
    print(results)

Impersonate 'chrome_133' does not exist, using 'random'


[{'title': 'Meaning of life', 'href': 'https://en.wikipedia.org/wiki/Meaning_of_life', 'body': 'The meaning of life is the concept of an individual\'s life, human life, or existence in general having an inherent significance or a philosophical point. There is no consensus on the specifics of such a concept, or whether the concept itself even exists in any objective sense. Thinking and discourse on the topic is sought in the English language through questions such as—but not limited to—"What is the meaning of life?", "What is the purpose of existence?", and "Why are we here?". There have been many proposed answers to these questions from many different cultural and ideological backgrounds. The search for life\'s meaning has produced much philosophical, scientific, theological, and metaphysical speculation throughout history. Different people and cultures believe different things for the answer to this question. Opinions vary on the usefulness of using time and resources in the pursuit o

In [7]:
from langchain_community.retrievers import ArxivRetriever

retriever = ArxivRetriever(
    top_k_results=1,          # how many papers to fetch
    doc_content_chars_max=4000  # limit per document (arXiv papers can be long!)
)

# Run a query
docs = retriever.invoke("LangGraph agents for tool use")

# Inspect results
for i, doc in enumerate(docs, 1):
    print(f"\n--- Paper {i} ---")
    print("Title:", doc.metadata.get("Title"))
    print("Authors:", doc.metadata.get("Authors"))
    print("Published:", doc.metadata.get("Published"))
    print("URL:", doc.metadata.get("Entry ID"))
    print("Content preview:\n", doc.page_content[:500])


--- Paper 1 ---
Title: AIRCC-Clim: a user-friendly tool for generating regional probabilistic climate change scenarios and risk measures
Authors: Francisco Estrada, Oscar Calderón-Bustamante, Wouter Botzen, Julián A. Velasco, Richard S. J. Tol
Published: 2021-10-30
URL: http://arxiv.org/abs/2111.01762v1
Content preview:
 Complex physical models are the most advanced tools available for producing realistic simulations of the climate system. However, such levels of realism imply high computational cost and restrictions on their use for policymaking and risk assessment. Two central characteristics of climate change are uncertainty and that it is a dynamic problem in which international actions can significantly alter climate projections and information needs, including partial and full compliance of global climate 


In [8]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
arxiv_query.invoke("Transformer models for NLP")

'Published: 2021-05-14\nTitle: A dissemination workshop for introducing young Italian students to NLP\nAuthors: Lucio Messina, Lucia Busso, Claudia Roberta Combei, Ludovica Pannitto, Alessio Miaschi, Gabriele Sarti, Malvina Nissim\nSummary: We describe and make available the game-based material developed for a laboratory run at several Italian science festivals to popularize NLP among young students.\n\nPublished: 2021-05-14\nTitle: Teaching NLP with Bracelets and Restaurant Menus: An Interactive Workshop for Italian Students\nAuthors: Ludovica Pannitto, Lucia Busso, Claudia Roberta Combei, Lucio Messina, Alessio Miaschi, Gabriele Sarti, Malvina Nissim\nSummary: Although Natural Language Processing (NLP) is at the core of many tools young people use in their everyday life, high school curricula (in Italy) do not include any computational linguistics education. This lack of exposure makes the use of such tools less responsible than it could be and makes choosing computational linguistic

In [9]:
## Wikipedia search tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wiki_query.invoke("Transformer models for NLP")

'Page: Lists of open-source artificial intelligence software\nSummary: These are lists include projects which release at least some of their software under open-source licenses and are related to artificial intelligence projects. These include software libraries, frameworks, platforms, and tools used for machine learning, deep learning, natural language processing, computer vision, reinforcement learning, artificial general intelligence, and more.\n\n\n\nPage: Generative pre-trained transformer\nSummary: A generative pre-trained transformer (GPT) is a type of large language model (LLM) that is widely used in generative AI chatbots. GPTs are based on a deep learning architecture called the transformer. They are pre-trained on large datasets of unlabeled content, and able to generate novel content.\nOpenAI was the first to apply generative pre-training to the transformer architecture, introducing the GPT-1 model in 2018. The company has since released many bigger GPT models. The chatbot 

### Writing a custom tools

In [18]:
from langchain.tools import tool

@tool
def get_personal_information(name: str) -> str:
    """"Use this tool to get personal information about Alice, Bob or Charlie, just provide their name as an input."""
    info = {
        "Alice": "Alice is a very nice person.",
        "Bob": "Bob is very clever.",
        "Charlie": "Charlie knows a lot of things."
    }
    return info.get(name, f"I don't know anything about {name}!")

@tool
def wiki_tool(query: str) -> str:
    """Use this tool to search Wikipedia for a query."""
    wiki_queery = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)


@tool
def arxiv_tool(query: str) -> str:
    """Use this tool to search ArXiv for a query."""
    arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)


## Tool Binding


In [26]:
tools = [get_personal_information, wiki_tool, arxiv_tool]
llm_with_tools = llm.bind_tools(tools)

In [31]:
response = llm_with_tools.invoke("Get me the personal information about Bob use get_personal_information tool.")
response.tool_calls
actual_tool_call = response.tool_calls[0]
get_personal_information.invoke(actual_tool_call["args"])

'Bob is very clever.'